# minilake + real Spark + real Delta Lake

This notebook writes a real Delta Lake table with a real Spark session, then reads the *same* files back through minilake's SQL Statement Execution API — exactly like a real Databricks workspace unifies DataFrame and SQL access to the same data.

Everything here is real: a real local Spark driver runs in this container, and the Delta files it writes live on a Docker volume shared with the `minilake` service.

## 1. Connect to minilake and create a catalog/schema

This resets minilake's state first, so the notebook can be re-run from scratch at any time.

In [ ]:
import requests

requests.post("http://minilake:8000/_minilake/reset")

from databricks.sdk import WorkspaceClient

w = WorkspaceClient(host="http://minilake:8000", token="dev")

w.catalogs.create(name="notebook_demo", comment="Created from JupyterLab")
w.schemas.create(name="events", catalog_name="notebook_demo")
print("catalog + schema ready")

## 2. Start a real Spark session with Delta Lake

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.appName("minilake-notebook")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

## 3. Register an EXTERNAL Delta table in minilake, write real data with Spark

`storage_location` is a path on the volume shared between this notebook and the `minilake` service — minilake only stores the *metadata* for EXTERNAL tables; the data itself is whatever real Delta files exist at that path.

In [ ]:
from databricks.sdk.service.catalog import DataSourceFormat, TableType

storage_location = "/data/delta/notebook_demo/events/people"

w.tables.create(
    name="people",
    catalog_name="notebook_demo",
    schema_name="events",
    table_type=TableType.EXTERNAL,
    data_source_format=DataSourceFormat.DELTA,
    storage_location=storage_location,
)

df = spark.createDataFrame([(1, "Alice"), (2, "Bob"), (3, "Charlie")], ["id", "name"])
df.write.format("delta").mode("overwrite").save(storage_location)
print("wrote a real Delta table")

## 4. Query the *same* data through minilake's SQL Statement Execution API

No copy, no sync step — minilake reads the real Delta files Spark just wrote (via DuckDB's `delta` extension).

In [ ]:
wh = w.warehouses.create(name="notebook_wh")

result = w.statement_execution.execute_statement(
    warehouse_id=wh.id,
    statement="SELECT * FROM notebook_demo.events.people ORDER BY id",
)
print(result.status.state)
print(result.result.data_array)

## 5. Round-trip: query with Spark too

Since it's a real Delta table, plain PySpark DataFrame reads work as well — same files, three access paths (Spark, minilake SQL, and Terraform/SDK-managed metadata).

In [ ]:
spark.read.format("delta").load(storage_location).show()